# Against the Odds

For each puzzle: **predict** and write your prediction in a Markdown cell, **simulate**,
**calculate** exactly, then **explain** any gap between your prediction and the
result. Implement the functions for [The office party](../problems/against-the-odds/office-party.md),
[Waiting for heads](../problems/against-the-odds/waiting-for-heads.md), or
[The 100 prisoners](../problems/against-the-odds/100-prisoners.md)
first; this notebook holds the investigations.

In [ ]:
import math
import random
from itertools import islice, product

import matplotlib.pyplot as plt
import pandas as pd

from pyproblems.birthday import (
    estimate_shared_birthday_probability,
    shared_birthday_probability,
    smallest_group_with_probability,
)
from pyproblems.coin_patterns import (
    average_waiting_time,
    coin_flips,
    expected_waiting_time,
    flips_until,
)
from pyproblems.prisoners import (
    clever_strategy,
    count_successes,
    cycle_lengths,
    estimate_survival,
    random_strategy,
    shuffled_drawers,
)

# Use a fixed seed for each comparison so that results can be repeated.
rng = random.Random(2026)

## The office party

Prediction: how large must a group be for a shared birthday to be more likely
than not? Why do you think so?

The plotting code is provided; your job is to fill in the tables of results.
Run the cells in order.

In [ ]:
def plot_probability(results, trials):
    """Plot a table with `people`, `simulated` and `exact` columns."""
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(results["people"], results["exact"], label="exact")
    ax.plot(
        results["people"],
        results["simulated"],
        "o",
        markersize=4,
        label=f"simulated ({trials:,} trials)",
    )
    ax.axhline(0.5, color="grey", linestyle=":")
    ax.set_xlabel("People in the group")
    ax.set_ylabel("Chance of a shared birthday")
    ax.legend()
    fig.tight_layout()
    return ax

In [ ]:
sizes = list(range(2, 61, 2))
trials = 100

results = pd.DataFrame({"people": sizes})

# TODO: add a "simulated" column with the estimated chance of a shared birthday
# for each group size (use `rng` and `trials`), and an "exact" column with the
# exact chance. Then run the next cell to plot them. The simulated points should
# hug the exact curve: how many trials does that take?

In [ ]:
ax = plot_probability(results, trials)
results

### Why so few?

Compare the number of pairs in the group, `math.comb(n, 2)`, with the number of
days. Then find the group size for even odds as the number of days grows.

In [ ]:
def plot_group_size(table):
    """Plot a table with `days` and `group_size` columns on log axes."""
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.loglog(
        table["days"],
        table["group_size"],
        "o-",
        label="smallest group for even odds",
    )
    ax.set_xlabel("Days in the year")
    ax.set_ylabel("Group size")
    ax.legend()
    fig.tight_layout()
    return ax

In [ ]:
days = [10, 100, 1_000, 10_000, 100_000]

table = pd.DataFrame({"days": days})

# TODO: add a "group_size" column: the smallest group with an even chance of a
# match for each number of days. Then add a "pairs" column with the number of
# pairs in that group (math.comb). Compare pairs with days: what do you notice?
# Run the next cell to plot group size against days.

In [ ]:
ax = plot_group_size(table)
table

### Uneven birthdays

Prediction: do uneven birthday popularities make a match more or less likely?

In [ ]:
# TODO: simulate groups whose birthdays are drawn with unequal weights using
# rng.choices(..., weights=...), and compare with the uniform case.

## Waiting for heads

Prediction: the average number of flips to see HH, and to see HT.

The plotting code is provided; your job is to fill in the tables of results.
Run the cells in order.

In [ ]:
def plot_waiting_times(table):
    """Plot a table with `pattern`, `simulated` and `exact` columns as bars."""
    ax = table.plot.bar(x="pattern", y=["simulated", "exact"], figsize=(9, 4))
    ax.set_xlabel("Pattern")
    ax.set_ylabel("Average flips until the pattern appears")
    return ax

In [ ]:
trials = 20_000

# TODO: estimate the average waiting time for "HH" and "HT" with
# average_waiting_time and `trials`, and compare them with your predictions and
# with expected_waiting_time.

In [ ]:
patterns = [
    "".join(flips) for length in (3, 4) for flips in product("HT", repeat=length)
]
table = pd.DataFrame({"pattern": patterns})

# TODO: add a "simulated" column (average_waiting_time, with 5_000 trials is
# plenty) and an "exact" column (expected_waiting_time) for every pattern. Then
# run the next cell to plot them. Which patterns are quick and which are slow,
# and what do the quick ones have in common?

In [ ]:
table = table.sort_values("exact")
ax = plot_waiting_times(table)
table

### Penney's game

You choose a three-flip pattern, then your opponent chooses a different one. The
pattern that appears first wins. Fill in the table of results; the plotting code
is provided.

In [ ]:
def plot_penney(results):
    """Plot a table with `first`, `second` and `first_win_rate` columns."""
    grid = results.pivot(index="first", columns="second", values="first_win_rate")
    fig, ax = plt.subplots(figsize=(6, 5))
    image = ax.imshow(grid, vmin=0, vmax=1, cmap="RdBu")
    ax.set_xticks(range(len(grid.columns)), grid.columns)
    ax.set_yticks(range(len(grid.index)), grid.index)
    ax.set_xlabel("Pattern chosen second")
    ax.set_ylabel("Pattern chosen first")
    for i, row in enumerate(grid.to_numpy()):
        for j, rate in enumerate(row):
            if rate == rate:
                ax.text(j, i, f"{rate:.2f}", ha="center", va="center", fontsize=8)
    fig.colorbar(image, label="How often the pattern chosen first wins")
    fig.tight_layout()
    return ax

In [ ]:
patterns = ["".join(flips) for flips in product("HT", repeat=3)]
games = 2_000

# TODO: build `results`, a table with one row for each of the 56 ordered pairs
# of different patterns and the columns "first", "second" and "first_win_rate".
# For each game, flip a coin many times to make one shared list of flips,
# list(islice(coin_flips(rng), 200)), and see which pattern appears first with
# flips_until. One way: collect a dictionary for each pair in a list, then call
# pd.DataFrame(rows). Then run the next cell to plot it. Is there a best first
# choice?

In [ ]:
ax = plot_penney(results)

## The 100 prisoners

Prediction: the chance that all 100 prisoners find their number if each opens
50 random drawers.

In [ ]:
# TODO: estimate the survival chance for random_strategy and clever_strategy with
# 10, 20, 50 and 100 prisoners, each allowed to open half the drawers.

### Alone and together

Compare how many prisoners find their number under each strategy, on average,
and then in distribution.

In [ ]:
# TODO: play many games with each strategy and record count_successes. Compare
# the averages, and plot a histogram of the counts for each strategy.

### Loops

Use cycle_lengths to explain who succeeds, then use it to estimate the survival
chance without simulating any prisoners.

In [ ]:
# TODO: for many arrangements, compare count_successes with the loop lengths.
# Then estimate the survival chance from the loops alone for 100 and 1,000
# prisoners, and see what it approaches.